<a href="https://colab.research.google.com/github/Hadron-JLeo/BachelorArbeit/blob/main/graph_hypercube.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Das Projekt wird auf Englisch und nach den Richtlinien von PEP8 programmiert

# Bibliotheken Imports

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field # Dataclass Klassen
from typing import ClassVar, NamedTuple, TypeAlias # Leichtere custom-Typisierungen

import numpy as np # Mathematik

from pprint import pprint # Schönere Print Ausgaben

#import open3d as o3d

# Klasse Punkt3D
## Implementierung von Datenstrukturen

In [ ]:
class Point3D(NamedTuple):
    """
    Repräsentiert einen Punkt im 3D-Raum.

    Attribute:
        x: x-Koordinate.
        y: y-Koordinate.
        z: z-Koordinate.
    """

    x: float
    y: float
    z: float


VertexId: TypeAlias = int
Color: TypeAlias = tuple[float, float, float]
PolygonVertices: TypeAlias = list[Point3D]
NeighbourIds: TypeAlias = set[VertexId]

# Klasse Polygon3D

In [ ]:
@dataclass(slots=True)
class Polygon3D:
    """
    Repräsentiert ein Polygon im 3D-Raum.

    Attribute:
        vertex_coords:
            Eckpunkte des Polygons in zyklischer Reihenfolge.
            Für Dimension d besitzt jedes Polygon d + 4 Eckpunkte.

        color:
            RGB-Farbe des Polygons, z. B. als tuple[float, float, float].

        hypercube_vertex_id:
            ID des zugehörigen Knotens im Hyperwürfelgraphen.

        neighbour_ids:
            IDs der benachbarten Hyperwürfelknoten.
            Für Dimension d besitzt jedes Polygon d Nachbarn.
    """

    vertex_coords: PolygonVertices
    color: Color
    hypercube_vertex_id: VertexId
    neighbour_ids: NeighbourIds = field(default_factory=set)

    def get_vertex_coords(self) -> PolygonVertices:
        """Gibt die Eckpunkte des Polygons zurück."""
        return self.vertex_coords

    def get_color(self) -> Color:
        """Gibt die Farbe des Polygons zurück."""
        return self.color

    def get_id(self) -> VertexId:
        """Gibt die ID des zugehörigen Hyperwürfelknotens zurück."""
        return self.hypercube_vertex_id

    def get_neighbour_ids(self) -> NeighbourIds:
        """Gibt die IDs der Nachbarpolygone zurück."""
        return self.neighbour_ids

    def get_centroid(self) -> Point3D:
        """
        Berechnet den Schwerpunkt der gespeicherten Eckpunkte.

        Return:
            Point3D: Durchschnitt aller Eckpunktkoordinaten.
        """
        count = len(self.vertex_coords)

        x_sum = sum(point.x for point in self.vertex_coords)
        y_sum = sum(point.y for point in self.vertex_coords)
        z_sum = sum(point.z for point in self.vertex_coords)

        return Point3D(
            x=x_sum / count,
            y=y_sum / count,
            z=z_sum / count,
        )

# Klasse HypercubeData

In [ ]:
@dataclass(slots=True, init=False)
class HypercubeData:
    """
    Speichert die Daten einer Hyperwürfel-Repräsentation.

    Singleton:
        Die Instanz wird nicht direkt durch HypercubeData(...) erzeugt,
        sondern über den HypercubeGenerator.

    Attribute:
        _dimension:
            Dimension d des Hyperwürfels.

        _polygons:
            Liste der Polygone.
            Für Dimension d gilt: Anzahl der Polygone = 2^d.
    """

    _dimension: int
    _polygons: list[Polygon3D]

    _instance: ClassVar[HypercubeData | None] = None

    # -- Ende Attribute -----------------------------------
    # -- Anfang Singleton-Logik ----------------------------

    def __init__(self, *args, **kwargs) -> None:
        """
        Verhindert direkte Instanzerzeugung.

        HypercubeData soll über HypercubeGenerator erzeugt werden.
        """
        raise RuntimeError(
            "HypercubeData darf nicht direkt erzeugt werden. "
            "Bitte HypercubeGenerator.generate_hypercube(...) verwenden."
        )

    @classmethod
    def _create_instance(
        cls,
        dimension: int,
        polygons: list[Polygon3D] | None = None,
    ) -> HypercubeData:
        """
        Erzeugt die Singleton-Instanz.

        Diese Methode ist absichtlich als interne Methode markiert.
        Sie soll nur vom HypercubeGenerator verwendet werden.

        Args:
            dimension:
                Dimension d des Hyperwürfels.

            polygons:
                Optionale Startliste von Polygonen.

        Return:
            HypercubeData: Die Singleton-Instanz.
        """
        if cls._instance is not None:
            if cls._instance.get_dimension() != dimension:
                raise ValueError(
                    "Es existiert bereits eine HypercubeData-Instanz "
                    f"mit dimension={cls._instance.get_dimension()}. "
                    f"Die neue Dimension {dimension} ist nicht erlaubt."
                )

            return cls._instance

        instance = object.__new__(cls)
        instance._dimension = dimension
        instance._polygons = polygons if polygons is not None else []

        cls._instance = instance
        return instance

    @classmethod
    def get_instance(cls) -> HypercubeData:
        """
        Gibt die bereits erzeugte Singleton-Instanz zurück.

        Wichtig:
            Die Instanz muss vorher über HypercubeGenerator erzeugt worden sein.

        Return:
            HypercubeData: Die vorhandene Singleton-Instanz.
        """
        if cls._instance is None:
            raise RuntimeError(
                "Es existiert noch keine HypercubeData-Instanz. "
                "Bitte zuerst HypercubeGenerator.generate_hypercube(...) aufrufen."
            )

        return cls._instance

    @classmethod
    def reset_instance(cls) -> None:
        """
        Setzt die Singleton-Instanz zurück.

        Diese Methode ist besonders in Jupyter praktisch,
        wenn man die Zellen mehrfach ausführt oder eine neue Dimension testen möchte.
        """
        cls._instance = None

    # -- Ende Singleton-Logik ------------------------------
    # -- Anfang Getter --------------------------------------

    def get_dimension(self) -> int:
        """Gibt die Dimension des Hyperwürfels zurück."""
        return self._dimension

    def get_hypercube_vertex_count(self) -> int:
        """
        Gibt die Anzahl der Knoten im Hyperwürfelgraphen zurück.

        Für einen d-dimensionalen Hyperwürfel gilt:
            Anzahl der Knoten = 2^d
        """
        return 2 ** self._dimension

    def get_polygon_count(self) -> int:
        """Gibt die aktuelle Anzahl gespeicherter Polygone zurück."""
        return len(self._polygons)

    def get_polygons(self) -> list[Polygon3D]:
        """Gibt alle Polygone zurück."""
        return self._polygons

    # -- Ende Getter ----------------------------------------
    # -- Anfang Setter --------------------------------------

    def set_polygons(self, polygons: list[Polygon3D]) -> None:
        """Setzt die Polygonliste."""
        self._polygons = polygons

    # -- Ende Setter ----------------------------------------
    # -- Weitere Methoden -----------------------------------

    def add_polygon(self, polygon: Polygon3D) -> None:
        """Fügt ein einzelnes Polygon hinzu."""
        self._polygons.append(polygon)

    def get_polygon_by_id(self, vertex_id: VertexId) -> Polygon3D | None:
        """
        Sucht ein Polygon anhand seiner Hyperwürfelknoten-ID.

        Args:
            vertex_id:
                ID des gesuchten Hyperwürfelknotens.

        Return:
            Polygon3D | None: Das gefundene Polygon oder None.
        """
        for polygon in self._polygons:
            if polygon.get_id() == vertex_id:
                return polygon

        return None

# Klasse HypercubeGenerator

In [ ]:
class HypercubeGenerator:
    """
    Erzeugt HypercubeData-Objekte und deren Polygone.

    Diese Klasse ist die zentrale Erzeugungsklasse.
    HypercubeData selbst wird nicht direkt instanziiert.
    """

    def generate_hypercube(self, dimension: int) -> HypercubeData:
        """
        Erzeugt eine Hyperwürfel-Repräsentation für die gegebene Dimension.

        Args:
            dimension:
                Dimension d des Hyperwürfels.

        Return:
            HypercubeData: Singleton-Datenobjekt mit erzeugten Polygonen.
        """
        polygons = self.generate_polygons(dimension)

        hypercube_data = HypercubeData._create_instance(
            dimension=dimension,
            polygons=polygons,
        )

        return hypercube_data

    def generate_polygons(self, dimension: int) -> list[Polygon3D]:
        """
        Erzeugt die Polygone für einen d-dimensionalen Hyperwürfel.

        Diese Methode ist zunächst nur ein Platzhalter.
        Der konkrete Rekursionsprozess wird später ergänzt.

        Args:
            dimension:
                Dimension d des Hyperwürfels.

        Return:
            list[Polygon3D]: Liste der erzeugten Polygone.
        """
        return []

## Test 1

In [ ]:
generator = HypercubeGenerator()

hypercube_data = generator.generate_hypercube(dimension=1)

print("Dimension:", hypercube_data.get_dimension())
print("Erwartete Knotenanzahl:", hypercube_data.get_hypercube_vertex_count())
print("Aktuelle Polygonanzahl:", hypercube_data.get_polygon_count())

Dimension: 1
Erwartete Knotenanzahl: 2
Aktuelle Polygonanzahl: 0


## Test 2

In [ ]:
from pprint import pprint


start_polygon = Polygon3D(
    vertex_coords=[
        Point3D(0.0, 0.0, 0.0),
        Point3D(0.0, 1.0, 0.0),
        Point3D(1.0, 1.0, 0.0),
        Point3D(1.0, 0.0, 0.0),
    ],
    color=(1.0, 0.5, 0.0),
    hypercube_vertex_id=0,
    neighbour_ids=set(),
)

polygon_info = {
    "polygon_id": start_polygon.get_id(),
    "color": start_polygon.get_color(),
    "vertex_count": len(start_polygon.get_vertex_coords()),
    "vertices": [
        {
            "x": point.x,
            "y": point.y,
            "z": point.z,
        }
        for point in start_polygon.get_vertex_coords()
    ],
    "centroid": {
        "x": start_polygon.get_centroid().x,
        "y": start_polygon.get_centroid().y,
        "z": start_polygon.get_centroid().z,
    },
    "neighbour_ids": sorted(start_polygon.get_neighbour_ids()),
}

pprint(polygon_info, sort_dicts=False)

{'polygon_id': 0,
 'color': (1.0, 0.5, 0.0),
 'vertex_count': 4,
 'vertices': [{'x': 0.0, 'y': 0.0, 'z': 0.0},
              {'x': 0.0, 'y': 1.0, 'z': 0.0},
              {'x': 1.0, 'y': 1.0, 'z': 0.0},
              {'x': 1.0, 'y': 0.0, 'z': 0.0}],
 'centroid': {'x': 0.5, 'y': 0.5, 'z': 0.0},
 'neighbour_ids': []}


## Test 3

Klassenmethode reset_instance wird verwendet um unser Singleton Objekt zurückzusetzen.

Dann wird ein neuer Hypercube generiert mit Dimension 3.

Anschließend werden die Daten in einem Dictionary zusammengefasst und ausgegeben.


In [ ]:
HypercubeData.reset_instance()

generator = HypercubeGenerator()
data = generator.generate_hypercube(dimension=3)

test_info = {
    "dimension": data.get_dimension(),
    "hypercube_vertex_count": data.get_hypercube_vertex_count(),
    "polygon_count": data.get_polygon_count(),
}

pprint(test_info, sort_dicts=False)

{'dimension': 3, 'hypercube_vertex_count': 8, 'polygon_count': 0}
